# Validation Tests — Part 2

This notebook tests generated-name validation and connector-specific checks.

| # | Scenario | Expected Result |
|---|----------|-----------------|
| 8 | Conflicting schedule in same pipeline group | `ValidationError` |
| 9 | Conflicting pause_status in same pipeline group | `ValidationError` |
| 10 | Conflicting connection_name in same pipeline group (SaaS) | `ValidationError` |
| 10b | Conflicting connection_name in same gateway (Database) | `ValidationError` |
| 11 | Cross-project pipeline group name collision | `ValidationError` |
| 13 | Mixed subgroup usage within a prefix | `ValidationError` |
| 14 | Missing project_name | `ConfigurationError` |
| 15 | Empty DataFrame | `ConfigurationError` |
| 16 | Invalid SCD type for connector | Warning, proceeds successfully |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%pip install pyyaml

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

username = w.current_user.me().user_name
workspace_host = w.config.host

WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'

In [ ]:
import sys
import os
import logging
import pandas as pd

sys.path.append(os.path.abspath('../../src'))

from tapworks.core.runner import run_pipeline_generation

# Enable logging to see warnings and info messages
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('tapworks')
logger.setLevel(logging.INFO)

targets = {
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
}

---
## Validation Point 2: Generated Names & Group Consistency
---

## Test 8: Conflicting Schedule in Same Pipeline Group

All 3 rows share `prefix=sales, subgroup=01` but row 2 has `*/30 * * * *` while others have `*/15 * * * *`.

**Expected:** `ValidationError` — conflicting schedule values in pipeline group.

In [ ]:
pd.read_csv('08_conflicting_schedule/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='08_conflicting_schedule/pipeline_config.csv',
        output_dir='08_conflicting_schedule/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 9: Conflicting pause_status in Same Pipeline Group

All 3 rows share `prefix=sales, subgroup=01` but row 2 has `UNPAUSED` while others have `PAUSED`.

**Expected:** `ValidationError` — conflicting pause_status values in pipeline group.

In [ ]:
pd.read_csv('09_conflicting_pause_status/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='09_conflicting_pause_status/pipeline_config.csv',
        output_dir='09_conflicting_pause_status/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 10: Conflicting connection_name in Same Pipeline Group (SaaS)

Uses Salesforce connector. Row 2 (Contact) has `sfdc_connection_b` while others have `sfdc_connection_a`.

**Expected:** `ValidationError` — conflicting connection_name in pipeline group.

In [ ]:
pd.read_csv('10_conflicting_connection_name/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='salesforce',
        input_source='10_conflicting_connection_name/pipeline_config.csv',
        output_dir='10_conflicting_connection_name/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 10b: Conflicting connection_name in Same Gateway (Database)

Uses SQL Server connector. Row 2 (Customers) has `sql_connection_b` while others have `sql_connection_a`. All share the same prefix/subgroup so they land in the same gateway.

**Expected:** `ValidationError` — conflicting connection_name in gateway.

In [ ]:
pd.read_csv('10b_conflicting_connection_name_db/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='10b_conflicting_connection_name_db/pipeline_config.csv',
        output_dir='10b_conflicting_connection_name_db/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 11: Cross-Project Pipeline Group Name Collision

Two projects (`project_alpha`, `project_beta`) both use `prefix=sales, subgroup=01`, producing the same pipeline group name.

**Expected:** `ValidationError` — pipeline group appears in multiple projects.

In [ ]:
pd.read_csv('11_cross_project_collision/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='11_cross_project_collision/pipeline_config.csv',
        output_dir='11_cross_project_collision/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

---
## Connector-Specific Validation
---

## Test 13: Mixed Subgroup Usage Within a Prefix

Prefix `sales` has row 1 with `subgroup=01`, row 2 with empty subgroup, and row 3 with `subgroup=02`.

**Expected:** `ValidationError` — mixed subgroup usage; all tables in a prefix must have explicit subgroups when any do.

In [ ]:
pd.read_csv('13_mixed_subgroup_usage/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='13_mixed_subgroup_usage/pipeline_config.csv',
        output_dir='13_mixed_subgroup_usage/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 14: Missing project_name

CSV has no `project_name` column and no default is provided.

**Expected:** `ConfigurationError` — missing required field: project_name.

In [ ]:
pd.read_csv('14_missing_project_name/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='14_missing_project_name/pipeline_config.csv',
        output_dir='14_missing_project_name/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 15: Empty DataFrame

CSV has headers only, no data rows.

**Expected:** `ConfigurationError` — input DataFrame is empty.

In [ ]:
pd.read_csv('15_empty_dataframe/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='15_empty_dataframe/pipeline_config.csv',
        output_dir='15_empty_dataframe/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 16: Invalid SCD Type for Connector

Rows 1-2 use invalid SCD types (`SCD_TYPE_3`, `INVALID`). Row 3 uses valid `SCD_TYPE_1`.

**Expected:** Warnings logged for invalid SCD types (ignored), pipeline generation **succeeds**.

In [ ]:
pd.read_csv('16_invalid_scd_type/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='16_invalid_scd_type/pipeline_config.csv',
        output_dir='16_invalid_scd_type/deployment',
        targets=targets,
    )
    print(f'SUCCESS: Processed {len(result)} rows (check logs above for SCD type warnings)')
    print(f'Pipeline groups: {result["pipeline_group"].nunique()}')
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')